In [1]:
from collections import defaultdict
from io import StringIO

import pandas as pd
import requests
from bs4 import BeautifulSoup, ResultSet

/Users/meenlike/.pyenv/versions/3.7.16/lib/python3.7/site-packages/requests/__init__.py:91: RequestsDependencyWarning: urllib3 (1.25.11) or chardet (5.1.0) doesn't match a supported version!
  RequestsDependencyWarning)


In [2]:
target_url = "https://en.wikipedia.org/wiki/List_of_The_Office_(American_TV_series)_episodes"
resp = requests.get(target_url)

resp.ok

True

In [3]:
soup = BeautifulSoup(resp.text, "html.parser")
raw_tables = soup.find_all('table',{'class':"wikitable"})

In [4]:
duplicate_counter = {}

special_symbols = ["‡", "†", "*"]

def splitted_episode(episode_title):
    return any([symbol in episode_title for symbol in special_symbols])


def rename_duplicates(row):
    row.Title = row.Title.replace("\"", "")
    title = row.Title

    if not splitted_episode(title):
        return row

    for symbol in special_symbols:
        title = title.replace(symbol, "")

    if title not in duplicate_counter:
        duplicate_counter[title] = 1

    else:
        duplicate_counter[title] += 1

    row.Title = f'{title}_part_{duplicate_counter[title]}'
    return row


## Парсинг таблиц серий сезонов

In [5]:
COLUMNS_NAMES = [
    "season",
    "episode",
    "title",
    "directed_by",
    "written_by",
    "air_date",
    "production_code",
]

def parse_season_table(season: int, raw: ResultSet) -> pd.DataFrame:
    df = pd.read_html(StringIO(str(raw[season])))[0].dropna(how="all")
    assert len(df.columns) == 8

    df = df.iloc[:, :-1]
    df.columns = COLUMNS_NAMES

    df.season = season
    df.episode = pd.to_numeric(df.episode, downcast='integer')
    df.production_code = pd.to_numeric(df.production_code, downcast='integer')

    # remove special symbols
    df.title = df.title.apply(
        lambda x: x.replace("\"", "")
                   .replace(":", "")
                   # .replace("*", "")
                   # .replace("‡", "")
                   # .replace("†", "")
                   
    )

    df.air_date = df.air_date.apply(
        lambda x: x.replace("\xa0", " ")
    )

    return df


# Symbols for splitted-episodes
special_symbols = ["‡", "†", "*"]


def splitted_episode(episode_title):
    """
    Check episode title to contain special symbols
    """
    return (
        any([symbol in episode_title for symbol in special_symbols])
    )


def add_part_postfix(title, count):
    return title + f" part {count}"


def remove_special_symbols(title) -> str:
    for symbol in special_symbols:
        title = title.replace(symbol, "")

    return title


manual_splitted = [
    "Goodbye, Michael†",
    "Livin' the Dream‡",
    "Moving On‡",
]


def as_json(df: pd.DataFrame):
    # counter
    splitted_episodes = defaultdict(int)

    # result
    episodes = list()

    for episode in df.T.to_dict().values():
        title: str = episode["title"]

        # single episode
        if not splitted_episode(title):
            episodes.append(episode)
            continue
    
        # maybe splitted episode
        clear_title = remove_special_symbols(title)

        if sum(df.title == title) == 2:
            splitted_episodes[clear_title] += 1
            episode["title"] = add_part_postfix(
                clear_title, splitted_episodes[clear_title]
            )
    
            episodes.append(episode)

        elif title in manual_splitted:
            # not splitted in wiki-table
            splitted_episodes[clear_title] += 1
            episode["title"] = add_part_postfix(
                clear_title, splitted_episodes[clear_title]
            )
    
            episodes.append(episode)

            second_part = episode.copy()

            splitted_episodes[clear_title] += 1            
            second_part["title"] = add_part_postfix(
                clear_title, splitted_episodes[clear_title]
            )

            episodes.append(second_part)

        else:
            episode["title"] = clear_title
            episodes.append(episode)

    return episodes


SyntaxError: invalid syntax (4237438115.py, line 25)

In [ ]:
data = list()

for season_no in range(1, 10):
    table = parse_season_table(season_no, raw_tables)
    data.extend(as_json(table))

In [ ]:
import json

with open("the_office/data/all_episodes.json", "w") as file:
    json.dump(data, file, indent=4)